In [7]:
import pandas as pd
import numpy as np
import json

PROCESSED = '../../data/processed'

def add_cyclical(df):
    df['hour_sin']  = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
    df['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
    return df

def process_and_save(src_csv, scale_cols, prefix):
    df = pd.read_csv(f'{PROCESSED}/{src_csv}', parse_dates=['datetime'], index_col='datetime')
    df = add_cyclical(df)
    for i in range(1, 25):
        df[f'load_lag{i}'] = df['load_mw'].shift(i)
    df = df.dropna()

    split = int(len(df) * 0.8)
    train, test = df.iloc[:split].copy(), df.iloc[split:].copy()
    print(f'Train: {len(train):,}  {train.index[0].date()} → {train.index[-1].date()}')
    print(f'Test:  {len(test):,}   {test.index[0].date()} → {test.index[-1].date()}')

    lag_scale = scale_cols + [f'load_lag{i}' for i in range(1, 25)]
    scaler = {col: {'mean': float(train[col].mean()), 'std': float(train[col].std())}
              for col in lag_scale}

    for col, s in scaler.items():
        train[col] = (train[col] - s['mean']) / s['std']
        test[col]  = (test[col]  - s['mean']) / s['std']

    train.to_csv(f'{PROCESSED}/{prefix}_train.csv')
    test.to_csv(f'{PROCESSED}/{prefix}_test.csv')
    with open(f'{PROCESSED}/{prefix}_scaler.json', 'w') as f:
        json.dump(scaler, f, indent=2)

    print(f'load_mw  mean={scaler["load_mw"]["mean"]:.1f} MW  std={scaler["load_mw"]["std"]:.1f} MW')
    print(f'Saved: {prefix}_train.csv, {prefix}_test.csv, {prefix}_scaler.json\n')

### 1. Averaged (combined_hourly)

In [8]:
SCALE_COLS_AVG = ['load_mw', 'temperature_c', 'precipitation_mm', 'solar_radiation_wm2',
                  'windspeed_ms', 'humidity_pct', 'gas_price_mmbtu']

process_and_save('combined_hourly.csv', SCALE_COLS_AVG, 'combined_hourly')

Train: 42,076  2019-01-02 → 2023-10-20
Test:  10,519   2023-10-20 → 2024-12-31
load_mw  mean=25225.2 MW  std=4789.3 MW
Saved: combined_hourly_train.csv, combined_hourly_test.csv, combined_hourly_scaler.json



### 2. Regional (combined_hourly_regional)

In [9]:
STATIONS     = ['sacramento', 'sanjose', 'fresno', 'la']
WEATHER_VARS = ['temperature_c', 'precipitation_mm', 'solar_radiation_wm2', 'windspeed_ms', 'humidity_pct']
WEATHER_COLS = [f'{s}_{v}' for s in STATIONS for v in WEATHER_VARS]
SCALE_COLS_REG = ['load_mw', 'gas_price_mmbtu'] + WEATHER_COLS

process_and_save('combined_hourly_regional.csv', SCALE_COLS_REG, 'combined_hourly_regional')

Train: 42,076  2019-01-02 → 2023-10-20
Test:  10,519   2023-10-20 → 2024-12-31
load_mw  mean=25225.2 MW  std=4789.3 MW
Saved: combined_hourly_regional_train.csv, combined_hourly_regional_test.csv, combined_hourly_regional_scaler.json

